[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_02_Prompt_Engineering.ipynb)

# 🧠 Lesson 02: Prompt Engineering
### AI/LLM/Agents Curriculum for Gourav

---

**What you'll learn today:**
- What prompt engineering actually is (and why it matters enormously)
- System prompts — shaping the LLM's identity and behavior
- Few-shot prompting — teaching by example
- Chain-of-thought (CoT) — making LLMs reason step-by-step
- XML structuring — clean, parseable prompts
- Output formatting — JSON, markdown, structured responses

**Prereq:** Lesson 01 (LLM Fundamentals). You should already know what an API call to Claude looks like.

---

## 🔑 One-Time Setup: Your API Key in Colab

Before running cells, store your Anthropic API key as a Colab Secret:
1. Click the **🔑 key icon** in the left sidebar
2. Add a secret named `ANTHROPIC_API_KEY`
3. Paste your API key as the value
4. Toggle "Notebook access" ON

You only need to do this once per Colab session.

In [ ]:
# 📦 Install dependencies
!pip install anthropic -q

import anthropic
import json
from google.colab import userdata

# Load API key from Colab Secrets
ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

# Helper: single-turn call (returns text)
def ask(prompt, system=None, model="claude-haiku-4-5-20251001", max_tokens=1024):
    """Simple helper that wraps the Anthropic API into one-liners."""
    kwargs = {"model": model, "max_tokens": max_tokens,
              "messages": [{"role": "user", "content": prompt}]}
    if system:
        kwargs["system"] = system
    response = client.messages.create(**kwargs)
    return response.content[0].text

print("✅ Setup complete! Let's engineer some prompts.")

---

## 🧩 Section 1: What Is Prompt Engineering?

When you call an LLM, **the prompt is your only input lever**. The model weights are frozen. You can't retrain the model mid-conversation. The only thing you control is what goes in — which makes prompt design critically important.

Think of it like this: the LLM is a very smart colleague who has read the entire internet. Your prompt is the briefing you give them before they start work. A vague briefing → vague work. A precise briefing → precise work.

Prompt engineering is the discipline of **designing inputs to get reliable, high-quality outputs** from LLMs.

### The anatomy of a typical prompt

```
┌───────────────────────────────┐
│  SYSTEM PROMPT                │  ← Who is the LLM? What are the rules?
│  (sets identity + behavior)   │
├───────────────────────────────┤
│  FEW-SHOT EXAMPLES (optional) │  ← Show it what good looks like
│  (in-context learning)        │
├───────────────────────────────┤
│  USER MESSAGE                 │  ← The actual task/question
│  (the task to perform)        │
└───────────────────────────────┘
```

Let's go section by section.

---

## 🎭 Section 2: System Prompts

A **system prompt** is a special message that appears *before* the conversation starts. It sets:
- The LLM's **persona** (who it is)
- Its **rules** (what it should/shouldn't do)
- Its **output format** (how it should respond)
- Background **context** (what it knows)

The system prompt is like a contract between you and the model. Every response the model gives will be shaped by it.

**Why this matters for agents:** When you build an AI agent, the system prompt is where you define the agent's capabilities, personality, and constraints. A customer support agent, a code reviewer, and a financial analyst are all the same base model — differentiated entirely by their system prompt.

Let's see the difference:

In [ ]:
# Without a system prompt — the model uses its default behavior
question = "What should I do if my code has a memory leak?"

print("=" * 60)
print("❌ WITHOUT system prompt:")
print("=" * 60)
response_default = ask(question)
print(response_default[:400], "...")

print()
print("=" * 60)
print("✅ WITH system prompt (Java expert, terse):")
print("=" * 60)

system_java = """
You are a senior Java engineer with 15 years of experience.
You give concise, opinionated answers with Java-specific examples.
Always mention relevant Java tools or libraries when applicable.
Respond in bullet points. Maximum 5 bullets.
"""

response_java = ask(question, system=system_java)
print(response_java)

In [ ]:
# 💡 EXPERIMENT: Try changing the persona entirely
# What if the system prompt makes it a Socratic teacher who only asks questions?

system_socratic = """
You are a Socratic tutor. You NEVER give direct answers.
Instead, you ask probing questions to help the student discover the answer themselves.
Keep each response to 2-3 questions only.
"""

print("🧠 Socratic mode:")
print(ask(question, system=system_socratic))

# 💡 Try your own system prompt below:
# system_mine = "You are a ..."
# print(ask(question, system=system_mine))

### 📝 System Prompt Best Practices

| Practice | Why it works |
|---|---|
| Start with a clear persona | Sets expectation for the entire response |
| Specify the output format | Avoids rambling or wrong structure |
| State what NOT to do | Boundary-setting is as important as instructions |
| Keep it focused | Long rambling system prompts dilute the signal |
| Put critical rules first | LLMs pay more attention to the beginning |

**Pro tip:** System prompts are processed at the start of every request. They count toward your token budget, so don't repeat context in every user message if it belongs in the system prompt.

---

## 🎯 Section 3: Few-Shot Prompting

**The core idea:** LLMs are incredible pattern matchers. If you show them examples of input → output pairs, they'll apply the same pattern to new inputs — even for tasks they've never been explicitly trained on.

This is called **in-context learning** (ICL). No fine-tuning needed. No retraining. Just examples in the prompt.

| Variant | Description |
|---|---|
| **Zero-shot** | No examples — just the task description |
| **One-shot** | One example before the actual task |
| **Few-shot** | 2–8 examples before the actual task |

Let's see why examples matter:

In [ ]:
# Task: Classify Java exceptions into categories

# ZERO-SHOT: just ask
zero_shot_prompt = """
Classify this Java exception into one of: [Runtime, Compile-time, IO, Concurrency]

Exception: NullPointerException
Category:"""

print("Zero-shot result:")
print(ask(zero_shot_prompt))

print()

# FEW-SHOT: same task, but with examples
few_shot_prompt = """
Classify the Java exception into one of: [Runtime, Compile-time, IO, Concurrency]
Respond with ONLY the category name, nothing else.

Exception: ClassCastException
Category: Runtime

Exception: FileNotFoundException
Category: IO

Exception: DeadlockException
Category: Concurrency

Exception: NullPointerException
Category:"""

print("Few-shot result:")
print(ask(few_shot_prompt))

In [ ]:
# More realistic use case: few-shot for custom output FORMAT
# The model learns your specific output schema from examples

system_analyzer = "You are a Java code analyzer. Analyze code and return structured insights."

few_shot_format_prompt = """
Analyze the Java code snippet and return structured feedback.

Example 1:
Code: for(int i=0; i<list.size(); i++) { ... }
Analysis:
  issue: Calling .size() on every iteration
  severity: medium
  fix: Cache list.size() in a variable before the loop

Example 2:
Code: String s = null; s.length();
Analysis:
  issue: Dereferencing null reference
  severity: critical
  fix: Add null check before calling .length()

Now analyze:
Code: HashMap map = new HashMap(); map.put("key", value);
Analysis:"""

print("📋 Few-shot format learning:")
print(ask(few_shot_format_prompt, system=system_analyzer))

### 💡 Few-Shot Tips

- **Quality > quantity**: 3 great examples beat 10 mediocre ones
- **Diversity**: Cover edge cases in your examples, not just the happy path  
- **Consistency**: Make sure every example follows the exact same format you want
- **Order matters a little**: Put your best example last (recency effect)
- **Token cost**: Each example costs tokens — balance quality vs cost for high-volume calls

---

## 🔗 Section 4: Chain-of-Thought (CoT) Prompting

LLMs are *next-token predictors*. By default, they jump straight to an answer. For simple questions, that's fine. For complex reasoning tasks, that's a problem — they often get the wrong answer because they skip steps.

**Chain-of-Thought (CoT)** prompting forces the model to write out its reasoning steps *before* giving the final answer. This dramatically improves accuracy on:
- Math and logic problems
- Multi-step reasoning  
- Code debugging
- Decision-making

**Intuition:** It's like asking someone to "show their work." The act of writing out intermediate steps forces the model to stay on track rather than guessing.

There are two flavors:
1. **Zero-shot CoT**: Just add "Think step by step" to the prompt
2. **Few-shot CoT**: Show examples with explicit reasoning chains

In [ ]:
# Problem: This kind of logic problem trips up LLMs without CoT
problem = """
A Java service processes 500 requests/second at peak.
Each request spawns 3 threads. Each thread uses 2MB of stack memory.
The JVM has 4GB of heap and 2GB reserved for thread stacks.
How many peak concurrent requests can the service handle before running out of thread stack memory?
"""

print("=" * 60)
print("❌ Without CoT (direct answer):")
print("=" * 60)
direct = ask(f"Answer this question with just a number: {problem}")
print(direct)

print()
print("=" * 60)
print("✅ With CoT (step-by-step reasoning):")
print("=" * 60)
cot = ask(f"{problem}\n\nThink step by step, showing your calculations. Then give the final answer.")
print(cot)

In [ ]:
# Few-shot CoT: Show reasoning examples, then the model follows the pattern

cot_prompt = """
You are a senior Java architect. Analyze design decisions step by step, then give a recommendation.

Example:
Question: Should I use HashMap or ConcurrentHashMap for a service shared across threads?
Reasoning:
  Step 1: HashMap is not thread-safe. Concurrent reads/writes cause ConcurrentModificationException.
  Step 2: ConcurrentHashMap uses segment locking — reads are lock-free, writes lock only the segment.
  Step 3: The service is shared across threads, so thread safety is required.
  Step 4: Performance impact of ConcurrentHashMap over HashMap is minimal for typical workloads.
Recommendation: Use ConcurrentHashMap.

Now answer:
Question: Should I use StringBuilder or String concatenation inside a loop that runs 10,000 times?
Reasoning:"""

print("🔗 Few-shot CoT in action:")
print(ask(cot_prompt))

### 🧠 Why CoT works (the intuition)

When an LLM generates text, each token is conditioned on all previous tokens. By writing out reasoning steps, the model creates a *scratchpad* of intermediate context that subsequent tokens are conditioned on.

```
Without CoT:  [problem] → [answer]           ← skips reasoning
With CoT:     [problem] → [step1] → [step2]   
              → [step3] → [answer]            ← each step grounds the next
```

**For agents specifically:** CoT is how you get agents to plan before they act. Instead of jumping straight to tool calls, a well-prompted agent reasons: "What do I know? What do I need to find out? Which tool should I call first?" — and this dramatically reduces wasted tool calls and errors.

---

## 📐 Section 5: XML Structuring

When prompts get complex — multiple pieces of context, examples, instructions — they become a wall of text that's hard for both humans and models to parse.

**XML tags** are the solution. Anthropic's models (Claude) are specifically trained to understand XML-tagged inputs and produce XML-tagged outputs. Think of XML tags as labeled boxes:

```xml
<context>...</context>        ← background info
<examples>...</examples>      ← few-shot examples
<task>...</task>              ← the actual task
<constraints>...</constraints> ← rules/limits
```

### Why this matters for agent systems

When you build multi-agent systems, you often need to:
1. Pass structured data *into* a prompt
2. Extract structured data *from* a response

XML tags make both directions clean and programmatically parseable.


In [ ]:
# Compare: unstructured vs XML-structured prompt for the same task

# ❌ Unstructured - hard to parse what's context vs task vs constraint
unstructured = """
Here is a Java method: public int calculate(int a, int b) { return a / b; }  
The context is that this is a production payment service and we've had 3 incidents caused by uncaught exceptions.
You should review the code and only focus on exception handling, not style issues. Tell me what's wrong.
"""

# ✅ XML-structured - every piece of information has a clear labeled container
structured = """
<context>
This is a production payment service.
We have had 3 incidents in the past month caused by uncaught exceptions.
</context>

<code>
public int calculate(int a, int b) {
    return a / b;
}
</code>

<constraints>
- Focus ONLY on exception handling issues
- Do NOT comment on code style or formatting
- List issues in order of severity (critical first)
</constraints>

<task>
Review the code and identify exception handling problems.
</task>
"""

system = "You are a Java code reviewer specializing in production safety."

print("❌ Unstructured prompt result:")
print("-" * 40)
print(ask(unstructured, system=system))

print()
print("✅ XML-structured prompt result:")
print("-" * 40)
print(ask(structured, system=system))

In [ ]:
# XML tags for OUTPUT extraction — get structured data back from the model

extract_prompt = """
<task>
Analyze the following Java stack trace. Extract key information and return it in XML tags.
</task>

<stack_trace>
Exception in thread "main" java.lang.NullPointerException
    at com.example.PaymentService.processPayment(PaymentService.java:47)
    at com.example.OrderController.checkout(OrderController.java:112)
    at sun.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
</stack_trace>

<output_format>
Return your analysis using these exact XML tags:
<exception_type>the exception class name</exception_type>
<root_cause_file>the most likely file causing the issue</root_cause_file>
<root_cause_line>line number</root_cause_line>
<likely_cause>one sentence explaining the probable cause</likely_cause>
<immediate_fix>one sentence fix recommendation</immediate_fix>
</output_format>
"""

response = ask(extract_prompt)
print("Raw XML response from model:")
print(response)

print()

# Now parse it programmatically!
import re

def extract_xml_tag(text, tag):
    match = re.search(f'<{tag}>(.*?)</{tag}>', text, re.DOTALL)
    return match.group(1).strip() if match else None

print("📦 Parsed into Python dict:")
parsed = {
    "exception": extract_xml_tag(response, "exception_type"),
    "file": extract_xml_tag(response, "root_cause_file"),
    "line": extract_xml_tag(response, "root_cause_line"),
    "cause": extract_xml_tag(response, "likely_cause"),
    "fix": extract_xml_tag(response, "immediate_fix"),
}
for k, v in parsed.items():
    print(f"  {k}: {v}")

### 🔑 Key Insight: XML enables agent pipelines

When you build agent systems, agents need to pass information to each other. XML-tagged outputs can be parsed by one agent and fed directly into the next agent's prompt. This is a foundational pattern in production agent architectures:

```
Agent 1 (Analyzer)
  → produces: <findings>...</findings><severity>critical</severity>

Agent 2 (Reporter) receives:
  <input_from_analyzer>...</input_from_analyzer>
  <task>Write an incident report based on the findings</task>
```

You'll use this pattern extensively in Lessons 4–6.

---

## 📤 Section 6: Output Formatting

By default, LLMs produce free-form text. But real applications need **structured, predictable outputs** that can be:
- Parsed by code
- Stored in databases
- Passed to other services
- Rendered in UIs

The three main patterns:

| Format | Use case |
|---|---|
| **JSON** | API responses, data pipelines, database storage |
| **Markdown** | Documentation, reports, human-readable outputs |
| **XML** | Agent-to-agent communication, structured extraction |

Let's see JSON in depth — it's the most commonly needed format in production.

In [ ]:
# Getting reliable JSON output

system_json = """
You are a code analysis API. You ALWAYS respond with valid JSON only.
Never include explanations, markdown code fences, or any text outside the JSON object.
"""

json_prompt = """
Analyze this Java method and return a JSON object with this exact schema:
{
  "method_name": "string",
  "complexity": "low|medium|high",
  "issues": ["issue1", "issue2"],
  "refactoring_needed": boolean,
  "estimated_fix_hours": number
}

Method to analyze:
public void saveUser(String name, String email, String phone, String address, 
                     String city, String country, String zip, Connection conn) throws Exception {
    Statement stmt = conn.createStatement();
    stmt.execute("INSERT INTO users VALUES ('" + name + "', '" + email + "')");
}
"""

raw_response = ask(json_prompt, system=system_json)
print("Raw response:")
print(raw_response)

print()

# Parse it
try:
    # Sometimes models wrap in ``` even when told not to — clean it up
    clean = raw_response.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    data = json.loads(clean)
    print("✅ Successfully parsed JSON:")
    print(f"  Method: {data['method_name']}")
    print(f"  Complexity: {data['complexity']}")
    print(f"  Issues found: {len(data['issues'])}")
    for issue in data['issues']:
        print(f"    → {issue}")
    print(f"  Refactoring needed: {data['refactoring_needed']}")
    print(f"  Estimated hours: {data['estimated_fix_hours']}")
except json.JSONDecodeError as e:
    print(f"❌ JSON parse error: {e}")
    print("Tip: Add 'Return ONLY the JSON, nothing else' to your prompt")

In [ ]:
# 🔒 Production pattern: Make JSON extraction bulletproof
# Use a <json>...</json> wrapper — then extract it

robust_prompt = """
<task>
Analyze this Java code snippet for security vulnerabilities.
</task>

<code>
String query = "SELECT * FROM accounts WHERE id = " + userId;
ResultSet rs = stmt.executeQuery(query);
</code>

<instructions>
Return your analysis as a JSON object wrapped in <json> tags.
Schema: {"vulnerability": string, "severity": "low|medium|high|critical", "cve_type": string, "remediation": string}
</instructions>
"""

response = ask(robust_prompt, system="You are a security code reviewer.")
print("Full response:")
print(response)

# Extract JSON from <json> tags — reliable even if model adds explanation text
json_match = re.search(r'<json>(.*?)</json>', response, re.DOTALL)
if json_match:
    parsed = json.loads(json_match.group(1).strip())
    print(f"\n✅ Parsed: severity={parsed['severity']}, type={parsed['cve_type']}")
else:
    print("\n⚠️ No <json> tags found — try more explicit formatting instructions")

---

## 🏗️ Section 7: Putting It All Together

Now let's combine everything into a production-grade prompt pattern:
**System prompt + XML structure + Few-shot CoT + JSON output**

This is close to what you'd write in a real agent.

In [ ]:
# A production-grade prompt combining all techniques

system_agent = """
You are CodeGuard, an automated Java code review agent for production services.
You analyze code changes and produce structured review reports.
You always reason step-by-step before giving your verdict.
You return all output as valid JSON wrapped in <review> tags.
"""

agent_prompt = """
<examples>
  <example>
    <code>int[] arr = new int[10]; arr[10] = 5;</code>
    <reasoning>Array has size 10, valid indices 0-9. Accessing index 10 is out of bounds.</reasoning>
    <review>{"verdict": "block", "severity": "critical", "issue": "ArrayIndexOutOfBoundsException at index 10", "fix": "Change index to 0-9 or increase array size"}</review>
  </example>
  <example>
    <code>List<String> names = new ArrayList<>(); Collections.sort(names);</code>
    <reasoning>Sorting an empty list is valid. No issues found.</reasoning>
    <review>{"verdict": "approve", "severity": "none", "issue": "", "fix": ""}</review>
  </example>
</examples>

<code_to_review>
public String getUserById(int id, Connection conn) {
    try {
        String sql = "SELECT name FROM users WHERE id = " + id;
        ResultSet rs = conn.createStatement().executeQuery(sql);
        return rs.getString("name");
    } catch (Exception e) {
        return null;
    }
}
</code_to_review>

<task>
Review this code. Think step-by-step about potential issues, then return your verdict.
Wrap your reasoning in <reasoning> tags and your JSON verdict in <review> tags.
</task>
"""

result = ask(agent_prompt, system=system_agent, max_tokens=1500)
print(result)

# Parse the structured output
review_match = re.search(r'<review>(.*?)</review>', result, re.DOTALL)
if review_match:
    review_data = json.loads(review_match.group(1).strip())
    print(f"\n{'🚫 BLOCKED' if review_data['verdict'] == 'block' else '✅ APPROVED'}")
    if review_data['issue']:
        print(f"Issue: {review_data['issue']}")
        print(f"Fix: {review_data['fix']}")

---

## 📚 Lesson Recap

You've now learned the core toolkit of prompt engineering:

| Technique | What it does | When to use it |
|---|---|---|
| **System prompts** | Defines agent identity, rules, format | Always — it's the foundation |
| **Few-shot** | Teaches output pattern via examples | When format or style must be exact |
| **Chain-of-thought** | Forces step-by-step reasoning | Complex reasoning, math, planning |
| **XML structuring** | Organizes complex prompt content | Multi-part prompts, agent pipelines |
| **Output formatting** | Gets parseable JSON/structured responses | Any code that needs to process output |

### 🔮 What's next: Lesson 3 — Tool Use / Function Calling

Right now, the LLM can only generate text. What if it could call functions, search the web, query a database, or run code? That's **tool use** — and it's the bridge from a chatbot to an actual agent. In Lesson 3, you'll give your LLM hands.

---

## 🧪 Exercises

Try these on your own before Lesson 3:

1. **System prompt experiment**: Write a system prompt that makes Claude respond only as a strict Java code reviewer who never writes code — only comments on it. Test it.

2. **Few-shot format drill**: Build a few-shot prompt that converts Java method signatures to OpenAPI YAML format. Give 2 examples, then test with a new method.

3. **CoT for bugs**: Give Claude a Java method with a subtle bug. First try asking it to find the bug directly. Then add "Think step by step." Compare the results.

4. **JSON pipeline**: Write a prompt that takes a Java class name and returns a JSON object with `{"package": "...", "typical_methods": [...], "common_pitfalls": [...]}`. Then write Python code to parse and pretty-print it.

In [ ]:
# 🧪 Exercise Starter — Your workspace
# Try any of the exercises above here

# Your code here:
my_system = """
TODO: Write your own system prompt
"""

my_prompt = """
TODO: Write your own prompt
"""

# response = ask(my_prompt, system=my_system)
# print(response)

print("✏️ Fill in my_system and my_prompt above, then uncomment the last two lines!")